# Exercice 04 – Corrigé


In [ ]:
import base64
import json
import time
from datetime import date
from pathlib import Path
from typing import Literal

import polars as pl
import pymupdf
import requests
from openai import OpenAI
from pydantic import BaseModel, Field

BASE_URL = "http://127.0.0.1:8080"
client = OpenAI(base_url=f"{BASE_URL}/v1", api_key="pas-besoin")

props = requests.get(f"{BASE_URL}/props").json()
print("Modalités :", props.get("modalities"))
print("Contexte  :", props["default_generation_settings"]["n_ctx"])

## Lire un PDF : texte éventuel + images de toutes les pages

In [ ]:
def lire_pdf(chemin: str, dpi: int = 110) -> dict:
    doc = pymupdf.open(chemin)
    texte = "\n".join(page.get_text() for page in doc)
    images = [page.get_pixmap(dpi=dpi).tobytes("png") for page in doc]
    return {"nb_pages": len(doc), "texte": texte, "scan": not texte.strip(), "images": images}


def image_part(png: bytes) -> dict:
    return {"type": "image_url",
            "image_url": {"url": "data:image/png;base64," + base64.b64encode(png).decode()}}


for f in ["contrats/contrat_01.pdf", "contrats/contrat_04.pdf"]:
    info = lire_pdf(f)
    print(f, "→", info["nb_pages"], "page(s),", "scan" if info["scan"] else f"natif ({len(info['texte'])} caractères)")

## Le schéma

In [ ]:
class Vehicule(BaseModel):
    marque: str
    modele: str
    carburant: Literal["essence", "diesel", "hybride", "hybride rechargeable", "électrique"]


class Assurance(BaseModel):
    type: Literal["omnium", "mini-omnium", "rc", "aucune"]
    compagnie: str = Field(description="Nom de la compagnie d'assurance. Chaîne vide si RC seule ou aucune assurance.")
    prime_mensuelle_eur: float = Field(description="Prime mensuelle en euros. 0 si aucune prime séparée.")


class Contrat(BaseModel):
    numero_contrat: str
    date_contrat: date = Field(description="Date du contrat au format AAAA-MM-JJ")
    agence: str = Field(description="Nom de la société de leasing")
    client: str = Field(description="Nom du client (preneur). Ce n'est pas le conseiller commercial.")
    vehicule: Vehicule
    duree_mois: int
    km_annuel: int
    acompte_eur: float = Field(description="Acompte ou premier loyer majoré. 0 s'il n'y en a pas.")
    mensualite_tvac_eur: float = Field(description="Mensualité TVA comprise. Si corrigée à la main, la valeur corrigée.")
    assurance: Assurance
    signature_client: bool = Field(description="True seulement si une signature manuscrite du client est visible")
    signature_agence: bool = Field(description="True seulement si une signature manuscrite de l'agence est visible")
    annule: bool = Field(description="True si le document porte un tampon ou une mention d'annulation")


SYSTEM = """Tu extrais les informations de contrats de leasing automobile belges à partir des images des pages.
Règles :
- Le client est aussi appelé « preneur ». Le conseiller commercial n'est jamais le client.
- La mensualité est aussi appelée « loyer » ou « redevance ». Donne toujours le montant TVA comprise (TVAC).
- Si un montant imprimé est barré et corrigé à la main, la valeur corrigée à la main est la bonne.
- Une signature est un tracé manuscrit. « Lu et approuvé » ou une date seule ne sont pas une signature.
- Montants en nombres décimaux (1.049,00 EUR → 1049.0). Dates au format AAAA-MM-JJ.
- Regarde toutes les pages : les signatures peuvent être sur la dernière."""

## Extraction d'un contrat

In [ ]:
def extraire(chemin: str) -> tuple[Contrat, dict]:
    pdf = lire_pdf(chemin)
    contenu = [{"type": "text", "text": f"Voici un contrat de {pdf['nb_pages']} page(s). Extrais les informations."}]
    contenu += [image_part(png) for png in pdf["images"]]

    debut = time.perf_counter()
    r = client.chat.completions.create(
        model="local",
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": contenu}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "contrat", "schema": Contrat.model_json_schema()}},
        temperature=0,
        max_tokens=1000,
    )
    if r.choices[0].finish_reason != "stop":
        raise ValueError(f"Réponse incomplète ({r.choices[0].finish_reason})")

    contrat = Contrat.model_validate_json(r.choices[0].message.content)
    stats = {"nb_pages": pdf["nb_pages"], "scan": pdf["scan"],
             "prompt_tokens": r.usage.prompt_tokens, "completion_tokens": r.usage.completion_tokens,
             "duree_s": round(time.perf_counter() - debut, 1)}
    return contrat, stats


contrat, stats = extraire("contrats/contrat_06.pdf")   # celui avec la mensualité corrigée à la main
print(stats)
contrat

## Les 20 contrats

In [ ]:
def aplatir(c: Contrat) -> dict:
    """Met le résultat au même format que la vérité terrain."""
    return {
        "numero_contrat": c.numero_contrat,
        "date_contrat": c.date_contrat.isoformat(),
        "agence": c.agence,
        "client": c.client,
        "marque": c.vehicule.marque,
        "modele": c.vehicule.modele,
        "carburant": c.vehicule.carburant,
        "duree_mois": c.duree_mois,
        "km_annuel": c.km_annuel,
        "acompte_eur": c.acompte_eur,
        "mensualite_tvac_eur": c.mensualite_tvac_eur,
        "assurance": c.assurance.type,
        "assurance_compagnie": c.assurance.compagnie,
        "assurance_prime_mensuelle_eur": c.assurance.prime_mensuelle_eur,
        "signature_client": c.signature_client,
        "signature_agence": c.signature_agence,
        "annule": c.annule,
    }


resultats = []
for chemin in sorted(Path("contrats").glob("*.pdf")):
    try:
        contrat, stats = extraire(str(chemin))
        ligne = {"fichier": chemin.name, **aplatir(contrat), **stats}
    except Exception as e:                      # on continue même si un contrat échoue
        ligne = {"fichier": chemin.name, "erreur": str(e)}
    resultats.append(ligne)
    print(f"{chemin.name}  {ligne.get('duree_s', '-')} s  {ligne.get('prompt_tokens', '-')} tokens  {ligne.get('erreur', '')}")

df = pl.DataFrame(resultats, infer_schema_length=None)   # lit toutes les lignes pour deviner les types
df.write_csv("resultats_contrats.csv")
print(f"\nTotal : {df['duree_s'].sum():.0f} s")
df

## Évaluation

In [ ]:
verite = pl.read_json("verite_terrain.json")

CHAMPS = ["numero_contrat", "date_contrat", "agence", "client", "marque", "modele", "carburant",
          "duree_mois", "km_annuel", "acompte_eur", "mensualite_tvac_eur", "assurance",
          "assurance_compagnie", "assurance_prime_mensuelle_eur",
          "signature_client", "signature_agence", "annule"]


def egal(attendu, obtenu) -> bool:
    if obtenu is None:                                # contrat en erreur : champ absent
        return False
    if isinstance(attendu, bool):
        return bool(obtenu) == attendu
    if isinstance(attendu, (int, float)):
        return abs(float(attendu) - float(obtenu)) < 0.01
    a, o = str(attendu).lower().strip(), str(obtenu).lower().strip()
    if a == "":                                       # rien d'attendu (ex. pas de compagnie)
        return o in ("", "-", "—", "aucune", "none")
    return bool(o) and (a in o or o in a)             # tolère « Mobilis Lease SA » pour « Mobilis Lease »


# Jointure vérité / prédictions : les colonnes prédites reçoivent le suffixe _pred
comp = verite.join(df, on="fichier", how="left", suffix="_pred")
lignes = comp.to_dicts()

scores = pl.DataFrame([
    {"fichier": l["fichier"], "format": l["format"], "mise_en_page": l["mise_en_page"],
     **{ch: egal(l[ch], l.get(f"{ch}_pred")) for ch in CHAMPS}}
    for l in lignes
])

par_champ = (scores.select(pl.col(CHAMPS).mean())
             .unpivot(variable_name="champ", value_name="taux")
             .sort("taux"))
with pl.Config(tbl_rows=20):
    print(par_champ)
print(f"Global : {par_champ['taux'].mean():.1%}")

In [ ]:
# Taux moyen sur tous les champs, par groupe
taux_ligne = pl.mean_horizontal(pl.col(CHAMPS).cast(pl.Float64))

print(scores.group_by("format").agg(taux_ligne.mean().alias("taux")).sort("format"))
print(scores.group_by("mise_en_page").agg(taux_ligne.mean().alias("taux")).sort("mise_en_page"))

In [ ]:
# Détail des erreurs, avec le piège prévu pour chaque contrat
erreurs = [
    {"fichier": l["fichier"], "champ": ch,
     "attendu": str(l[ch]), "obtenu": str(l.get(f"{ch}_pred")),   # str : types mélangés dans une même colonne
     "pieges": l["pieges"]}
    for l, s in zip(lignes, scores.to_dicts())
    for ch in CHAMPS if not s[ch]
]

with pl.Config(fmt_str_lengths=80, tbl_rows=100):
    display(pl.DataFrame(erreurs))

## Éléments d'analyse

Ce qu'on s'attend à observer ; les chiffres exacts dépendent du modèle et de la version de llama.cpp, à confirmer avec vos résultats.

- **Champs imprimés** (numéro, client, véhicule, durée) : quasi parfaits sur les natifs, légèrement moins sur les scans, surtout le scan flou (15) et le scan incliné (08). Les erreurs éventuelles portent plutôt sur des caractères proches (`1`/`7`, `O`/`0`) dans le numéro de contrat.
- **Mensualité** : les pièges sont les corrections manuscrites (06, 17) et le double affichage HTVA/TVAC de la mise en page B. La règle du system prompt et la `description` du champ font une vraie différence : retirez-les et comparez.
- **Signatures** : le champ le plus difficile. Le contrat 10 (« Lu et approuvé » sans signature) est le candidat n° 1 à un faux « signé ». Les signatures en page 2 (mise en page C) sont trouvées seulement si les deux pages sont envoyées.
- **Annulé** : le tampon est gros et visible, il est en général détecté.
- **Natif vs scan** : l'écart est modeste avec un bon modèle vision, parce qu'on envoie des images dans les deux cas. Il serait énorme avec une extraction purement textuelle : `get_text()` ne renvoie rien sur les 7 scans, et sur les natifs il ne voit ni les signatures, ni les corrections manuscrites, ni le tampon.
- **Coût** : regardez `prompt_tokens` : les contrats de deux pages coûtent deux fois plus, et le temps est dominé par le traitement du prompt (l'encodage des images), pas par la génération.

## Pour aller plus loin

- Envoyer **aussi** le texte extrait pour les PDF natifs : les champs imprimés s'améliorent, mais la mensualité des contrats 06 et 17 risque d'empirer (le texte ne contient que le montant barré).
- Refaire l'extraction à 72 puis 150 dpi : tokens, temps, taux de réussite.
- Deux appels par contrat : un pour les champs imprimés, un second, dédié, pour les signatures et l'annulation.